# Global Trends — Freedom in the World

**Notebook 04 of 08**

### Purpose

The cleaning is done. From here the notebooks analyse the **long analytical dataset** (`data/processed/freedom_in_world_long.csv`). This notebook asks the first global question: how have freedom scores evolved between 2013 and 2026 across all 197 economies, and which indicators moved most?

### Scope

The overall score `FH_FIW_TOTAL` (0–100) is the natural starting point — it is on a single common scale, so comparisons are clean. Other scales (political rights 0–40, civil liberties 0–60, question-level 0–4) enter only where we explicitly normalize to the scale width.

## Data and Inputs

| Item | Location |
|---|---|
| Long analytical dataset | `data/processed/freedom_in_world_long.csv` |

**Inputs:** produced by notebook 03 — 110,320 rows: economy, indicator (code, label, category, scale), year, score.

**Questions this notebook answers:**
1. How have Freedom in the World scores changed over time?
2. What does the distribution of scores look like each year?
3. Which indicators show the largest changes?
4. How many economies improved or deteriorated?
5. Which years show notable changes?

**Method:** aggregate the overall score, then answer each question with a table or an interactive Plotly chart. Charts come from the reusable helpers in `src/visualizations.py`.

## Setup: imports and the project root

Same bootstrap as the previous notebooks. New here: the visualization helpers `create_trend_chart()` and `create_heatmap()` from `src/visualizations.py`.

In [1]:
import sys
from pathlib import Path

current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd
import plotly.express as px

from src.data_loader import load_processed_data
from src.visualizations import create_trend_chart, create_heatmap

print('Imports ready.')

Imports ready.


### Interpretation

Setup ran cleanly. The rest of the notebook consumes only the processed dataset — the raw CSV is not touched again in the analysis phase.

## 1. Load the analytical dataset

**Question:** is the long dataset ready to analyse?

**Method:** load it with the shared helper and preview.

In [2]:
long = load_processed_data()
print('Long dataset shape:', long.shape)
long.head()

Long dataset shape: (110320, 9)


,REF_AREA,Economy,INDICATOR,INDICATOR_LABEL,Category,UNIT_MEASURE,UNIT_MEASURE_LABEL,Year,Score
0,COD,"Congo, Dem. Rep.",FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law,0_TO_4,0-4 scale,2013,0.0
1,MYS,Malaysia,FH_FIW_F3,Rule of Law: Is there protection from the ille...,Rule of Law,0_TO_4,0-4 scale,2013,1.0
2,TZA,Tanzania,FH_FIW_F4,"Rule of Law: Do laws, policies, and practices ...",Rule of Law,0_TO_4,0-4 scale,2013,3.0
3,TZA,Tanzania,FH_FIW_G2,Personal Autonomy And Individual Rights: Are i...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale,2013,2.0
4,BEL,Belgium,FH_FIW_G3,Personal Autonomy And Individual Rights: Do in...,Personal Autonomy And Individual Rights,0_TO_4,0-4 scale,2013,4.0


### Interpretation

The dataset loads as saved in notebook 03: 110,320 rows and 9 columns. `Year` reads back as an integer and the descriptors as strings — exactly the contract notebook 03 established.

## 2. Focus on the overall score

**Question:** how do we isolate the headline indicator?

**Method:** filter to `FH_FIW_TOTAL` and coerce its scores to numeric (the column is mixed-type because the categorical STATUS rows share it).

In [3]:
total = long[long['INDICATOR'] == 'FH_FIW_TOTAL'].copy()
total['Score'] = pd.to_numeric(total['Score'], errors='coerce')

print('TOTAL rows:', len(total), '| years:', total['Year'].min(), '-', total['Year'].max())
print('Economies with a TOTAL score in 2026:', total[total['Year'] == 2026]['Score'].notna().sum())

TOTAL rows: 2758 | years: 2013 - 2026
Economies with a TOTAL score in 2026: 196


### Interpretation

2758 rows = 197 economies × 14 years of overall scores. All but a handful of 2026 scores are present. Missing cells are left missing — the aggregations below skip them, so the trend is never distorted by imputation.

## 3. Question 1 — the global mean per year

**Question:** how has the average overall score moved year by year?

**Method:** take the mean of the overall score across all economies for each year.

In [4]:
yearly_mean = total.groupby('Year')['Score'].mean().round(2)
yearly_mean

Year
2013    61.23
2014    61.19
2015    60.96
2016    60.68
2017    59.99
2018    59.60
2019    59.31
2020    59.03
2021    58.55
2022    57.97
2023    57.78
2024    57.44
2025    57.15
2026    56.90
Name: Score, dtype: float64

### Interpretation

**The global average declined in every single year** — a striking, unbroken pattern. It fell from **61.23 (2013) to 56.90 (2026)**, a drop of **4.33 points** on the 0–100 scale over 14 years. No single year reversed the direction.

## 4. Question 1, visualized — the global trend line

**Question:** how does the decline look as a chart?

**Method:** plot the yearly means with the shared `create_trend_chart()` helper.

In [5]:
fig = create_trend_chart(
    yearly_mean.reset_index(),
    title='Global average overall freedom score, 2013-2026',
    y_label='Mean overall score (0-100)',
)
fig.show()

### Interpretation

The line falls steadily and almost linearly. There is no single crash year — the erosion is a **slow, continuous drift** rather than a sudden event. The largest single step-down comes in 2017 (examined in section 8).

## 5. Question 2 — the distribution each year

**Question:** is the decline shared broadly, or driven by a few economies?

**Method:** box plots of the overall score per year — they show the median, the spread, and the tails.

In [6]:
fig = px.box(
    total,
    x='Year',
    y='Score',
    title='Distribution of overall freedom scores by year, 2013-2026',
    labels={'Score': 'Overall score (0-100)', 'Year': 'Year'},
)
fig.update_layout(template='plotly_white', title_x=0.5)
fig.show()

### Interpretation

The whole distribution shifts down, not just the tail: the **median fell from 64 (2013) to 62.5 (2026)**, and the bottom of the range sank from a minimum of 2 to a minimum of 0 (South Sudan and Sudan at the floor). The top stays capped at 100 — Norway and Sweden — so the decline is compression from below and the middle, not a narrowing at the top.

## 6. Question 3 — which indicators changed most?

**Question:** beyond the headline score, which specific dimensions deteriorated or improved?

**Method:** compare each indicator's mean score between 2013 and 2026. Because indicators live on different scales (0–4 questions up to 0–100 totals), the change is expressed **as a percentage of the scale width** — a −0.5 move on a 0–4 question (−12.5%) is comparable to a −12.5 move on the 0–100 total (−12.5%). The inverted 1–7 ratings and the categorical STATUS are excluded here; they get their own treatment in notebook 07.

In [7]:
sm_map = {'0_TO_4': 4, '0_TO_12': 12, '0_TO_16': 16, '0_TO_40': 40, '0_TO_60': 60, '0_TO_100': 100}

numeric = long[long['UNIT_MEASURE'].isin(sm_map)].copy()
numeric['Score'] = pd.to_numeric(numeric['Score'], errors='coerce')

s2013 = numeric[numeric['Year'] == 2013].set_index(['INDICATOR', 'REF_AREA'])['Score']
s2026 = numeric[numeric['Year'] == 2026].set_index(['INDICATOR', 'REF_AREA'])['Score']
paired = pd.concat([s2013, s2026], axis=1, keys=['s2013', 's2026']).dropna()
paired.head()

s2013  s2026
INDICATOR REF_AREA              
FH_FIW_F3 COD         0.0    0.0
          MYS         1.0    2.0
FH_FIW_F4 TZA         3.0    1.0
FH_FIW_G2 TZA         2.0    2.0
FH_FIW_G3 BEL         4.0    4.0

In [8]:
change = paired.groupby('INDICATOR').apply(lambda g: (g['s2026'] - g['s2013']).mean())
scale_max = numeric[['INDICATOR', 'UNIT_MEASURE']].drop_duplicates()
scale_max = scale_max.assign(scale_max=scale_max['UNIT_MEASURE'].map(sm_map)).set_index('INDICATOR')['scale_max']

movers = pd.DataFrame({'mean_delta_points': change.round(2)})
movers['mean_delta_pct_of_scale'] = (change / scale_max[change.index] * 100).round(1)
movers['abs_pct'] = movers['mean_delta_pct_of_scale'].abs()
movers.sort_values('abs_pct', ascending=False).head(10)

,mean_delta_points,mean_delta_pct_of_scale,abs_pct
INDICATOR,,,
FH_FIW_D4,-0.51,-12.8,12.8
FH_FIW_C1,-0.28,-6.9,6.9
FH_FIW_D,-1.02,-6.3,6.3
FH_FIW_A1,-0.24,-6.0,6.0
FH_FIW_A3,-0.24,-6.0,6.0
FH_FIW_A,-0.71,-5.9,5.9
FH_FIW_A2,-0.23,-5.7,5.7
FH_FIW_D3,-0.23,-5.7,5.7
FH_FIW_B2,-0.22,-5.6,5.6


### Interpretation

The deterioration is **broad-based**: 35 of the 37 numeric indicators moved down on average between 2013 and 2026. Only two improved — `FH_FIW_G3` (personal social freedoms, +0.3% of scale) and `FH_FIW_ADD_Q` (additional question, +1.4%).

The largest declines concentrate in **freedom of expression and the electoral process**: `FH_FIW_D4` (a freedom-of-expression question) fell furthest at **−12.8% of its scale**, followed by `FH_FIW_C1` (functioning of government, −6.9%), the freedom-of-expression subtotal `FH_FIW_D` (−6.3%), and the electoral-process questions `FH_FIW_A1`/`FH_FIW_A3` (both −6.0%). Rule-of-law questions (`FH_FIW_F2`, −5.5%) also declined.

## 7. Question 4 — how many economies improved or deteriorated?

**Question:** is the global decline universal, or do some economies go against the trend?

**Method:** for each economy, compare the overall score in 2026 with 2013; count improvers, decliners and unchanged; then list the extremes.

In [9]:
y2013 = total[total['Year'] == 2013].set_index(['REF_AREA', 'Economy'])['Score']
y2026 = total[total['Year'] == 2026].set_index(['REF_AREA', 'Economy'])['Score']
both_years = pd.concat([y2013, y2026], axis=1, keys=['s2013', 's2026']).dropna()

improved = (both_years['s2026'] > both_years['s2013']).sum()
deteriorated = (both_years['s2026'] < both_years['s2013']).sum()
unchanged = (both_years['s2026'] == both_years['s2013']).sum()

print('Improved (2026 > 2013):', improved)
print('Deteriorated (2026 < 2013):', deteriorated)
print('Unchanged:', unchanged)
print('Missing either year:', 197 - len(both_years))

Improved (2026 > 2013): 52
Deteriorated (2026 < 2013): 128
Unchanged: 16
Missing either year: 1


In [10]:
both_years['change'] = (both_years['s2026'] - both_years['s2013']).round(1)
print('Mean change across economies:', both_years['change'].mean().round(2))
print()
print('Largest deteriorations:')
print(both_years.nsmallest(5, 'change')[['s2013', 's2026', 'change']])
print()
print('Largest improvements:')
print(both_years.nlargest(5, 'change')[['s2013', 's2026', 'change']])

Mean change across economies: -4.19

Largest deteriorations:
                       s2013  s2026  change
REF_AREA Economy                           
TZA      Tanzania       66.0   28.0   -38.0
NIC      Nicaragua      51.0   14.0   -37.0
SLV      El Salvador    77.0   42.0   -35.0
LBY      Libya          43.0   10.0   -33.0
BFA      Burkina Faso   53.0   20.0   -33.0

Largest improvements:
                      s2013  s2026  change
REF_AREA Economy                          
FJI      Fiji          37.0   72.0    35.0
GMB      Gambia, The   23.0   51.0    28.0
BTN      Bhutan        46.0   69.0    23.0
LKA      Sri Lanka     43.0   63.0    20.0
XKX      Kosovo        42.0   61.0    19.0


In [11]:
counts = pd.DataFrame({'outcome': ['Improved', 'Unchanged', 'Deteriorated'],
                       'economies': [improved, unchanged, deteriorated]})
fig = px.bar(
    counts,
    x='outcome',
    y='economies',
    color='outcome',
    text='economies',
    title='How many economies improved, stayed the same, or deteriorated (2013 to 2026)',
    labels={'outcome': 'Outcome', 'economies': 'Number of economies'},
)
fig.update_layout(template='plotly_white', title_x=0.5, showlegend=False)
fig.show()

### Interpretation

More than twice as many economies **deteriorated (128) as improved (52)**; 16 were unchanged and one lacks a score in one of the two years. The average economy lost **4.2 points**.

The deepest declines are severe: Tanzania **−38**, Nicaragua **−37**, El Salvador **−35**, Libya **−33**, Burkina Faso **−33**. The biggest gains are far smaller in scale: Fiji **+35**, Gambia **+28**, Bhutan **+23**, Sri Lanka **+20**, Kosovo **+19**. The global fall is driven by many deep national declines, while few countries improved strongly.

## 8. Question 5 — which years are notable?

**Question:** were any years turning points?

**Method:** take the year-over-year change of the global mean.

In [12]:
yoy = yearly_mean.diff().round(2)
yoy_table = pd.DataFrame({'global_mean': yearly_mean, 'yoy_change': yoy})
yoy_table

,global_mean,yoy_change
Year,,
2013,61.23,NaN
2014,61.19,-0.04
2015,60.96,-0.23
2016,60.68,-0.28
2017,59.99,-0.69
2018,59.60,-0.39
2019,59.31,-0.29
2020,59.03,-0.28
2021,58.55,-0.48


### Interpretation

**Every year-on-year change is negative** — there was no year in which the global average rose. The sharpest annual decline is **2017 (−0.69 points)**, the mildest 2014 (−0.04). The pattern is remarkable for its consistency: thirteen consecutive annual declines of similar size (roughly −0.2 to −0.7 points) rather than one shock year.

## 9. Indicator × year heatmap

**Question:** can we see the breadth of the decline at a glance?

**Method:** for each indicator, take the mean score per year as a **percentage of the indicator's scale**, and plot indicator × year as a heatmap (using the shared helper). Rows are labelled with category and code.

In [13]:
heat = numeric.copy()
heat['pct_of_scale'] = heat['Score'] / heat['UNIT_MEASURE'].map(sm_map) * 100

pivot = heat.pivot_table(index='INDICATOR', columns='Year', values='pct_of_scale', aggfunc='mean')

labels = numeric[['INDICATOR', 'Category']].drop_duplicates()
labels['short'] = labels['Category'] + ' (' + labels['INDICATOR'].str.replace('FH_FIW_', '') + ')'
pivot.index = pivot.index.map(labels.set_index('INDICATOR')['short'])

fig = create_heatmap(
    pivot,
    title='Average score as share of scale, indicator x year (2013-2026)',
    colorbar_label='% of scale',
)
fig.show()

### Interpretation

The heatmap makes the breadth visible: a wide band of darker cells (lower values) sweeps across most indicators toward the right-hand years. The darkest block sits in **Freedom of Expression and Belief (D)**, with `D4` at the bottom of its scale region, while the civil-liberties totals (PR/CL rows) hold more of their value. The only bright (improving) patches are the small ones for `G3` and `ADD_Q`. What the headline score hides is that expression-related rights, not all rights, drove the global fall.

## Summary and next question

### What we learned

- The global average overall score fell **every year** from 61.23 (2013) to 56.90 (2026) — a −4.33-point drift with no rebound.
- The distribution shifted down as a whole (median 64 → 62.5); **128 economies deteriorated** versus 52 that improved.
- The decline is broad-based across indicators but led by **freedom of expression** (`FH_FIW_D4`, −12.8% of scale) and the electoral process; only two indicators improved.
- Every year-on-year change was negative; the sharpest step was **2017 (−0.69)**.
- These are observations of the FiW scores, not causal claims — explaining the decline is out of scope for this dataset.

### Next question

*How do individual economies fit into this global picture?* — notebook 05, country analysis, zooms into selected economies (e.g. the big movers Tanzania, Fiji, Nicaragua) and their indicator profiles.